In [15]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [16]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [17]:
print(x_train.shape)
print(x_test.shape)
print(type(x_train))
print()
print(x_train)

(60000, 28, 28)
(10000, 28, 28)
<class 'numpy.ndarray'>

[[[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 ...

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]]


In [18]:
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0             # since layers.Conv2D expects a 4D tensor, hence passing a 2D tensor will cause error
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

In [19]:
print(x_train.shape)

(60000, 28, 28, 1)


In [20]:
import optuna

In [ ]:
def my_model(trial):
    lr = trial.suggest_float('lr', 1e-8, 1e-2, log=True)
    num_conv_layers = trial.suggest_int('n_trials', 1, 3, step=1)
    
    model = keras.Sequential([
        keras.Input(shape=(28,28,1))
    ])

    for i in range(num_conv_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 16, 768, step=16)
        activation_fn = trial.suggest_categorical(f'activation_fn_{i}', ['relu', 'tanh', 'sigmoid', 'elu', 'leaky_relu'])
        padding = trial.suggest_categorical(f'padding_{i}', ['valid', 'same'])
        dropouts = trial.suggest_float(f'dropouts_{i}', 0.0,0.5, step=0.1)
              
        
        kernel_initializer = trial.suggest_categorical(f'kernel_initializer_{i}', ['he_normal', 'glorot_normal'])
        regularizer = trial.suggest_categorical(f'regularizer_{i}', ['l1','l2'])
        reg_rate = trial.suggest_float(f'reg_rate_{i}', 1e-8, 1e-2, log=True)

        if regularizer == 'l1':
            chosen_regularizer = regularizers.l1(reg_rate)
        else:
            chosen_regularizer = regularizers.l2(reg_rate)

        

        bias_initializer = trial.suggest_categorical(f'bias_initializer_{i}', ['he_normal', 'glorot_normal', 'he_uniform', 'glorot_uniform'])
        bias_regularizer = trial.suggest_categorical(f'bias_regularizer_{i}', ['l1','l2'])
        bias_reg_rate = trial.suggest_float(f'bias_reg_rate_{i}', 1e-8, 1e-2, log=True)

        if bias_regularizer == 'l1':
            chosen_bias_regularizer = regularizers.l1(bias_reg_rate)
        else:
            chosen_bias_regularizer = regularizers.l2(bias_reg_rate)


        model.add(layers.Conv2D(neurons, 3, activation=activation_fn, padding=padding, kernel_initializer=kernel_initializer, kernel_regularizer=chosen_regularizer, bias_initializer=bias_initializer, bias_regularizer=chosen_bias_regularizer))
        if dropouts > 0.0:
            model.add(layers.SpatialDropout2D(dropouts))
        
        model.add(layers.MaxPooling2D(pool_size=(2,2)))

                  
    temp_neurons = trial.suggest_int(f'temp_neurons', 16, 256, step=16)
    temp_activation_fn = trial.suggest_categorical('temp_activation_fn', ['relu', 'tanh', 'sigmoid', 'elu', 'leaky_relu'])
    
    model.add(layers.Flatten())
    model.add(layers.Dense(temp_neurons, activation=temp_activation_fn))
    model.add(layers.Dense(10, activation='softmax'))

    
    epsilon = trial.suggest_float('epsilon', 1e-7, 1e-2, log=True)
    model.compile(
        loss = keras.losses.SparseCategoricalCrossentropy(),
        optimizer = keras.optimizers.Adam(
            lr,
            beta_1 = 0.9,
            beta_2 = 0.999,
            epsilon = epsilon
        ),
        metrics = ['accuracy']
    )


    early_stop = EarlyStopping(
        patience=3,
        verbose = 1,
        restore_best_weights = True,
        monitor = 'val_accuracy'
    )

    history = model.fit(
        x_train, y_train,
        callbacks = [early_stop],
        epochs = 5,
        validation_split = 0.2,
        batch_size = 128,
        verbose = 1
    )


    model.save(f"model_trial_{trial.number}.keras")

    return max(history.history['val_accuracy'])


study = optuna.create_study(direction='maximize')
study.optimize(my_model, n_trials=10)


[I 2026-09-25 10:34:39,085] A new study created in memory with name: no-name-179dc605-6720-4309-b788-9692f6c13b3f


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.1239 - loss: 8.9250 - val_accuracy: 0.2842 - val_loss: 8.8038
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.3421 - loss: 8.7056 - val_accuracy: 0.5947 - val_loss: 8.5094
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.5405 - loss: 8.3608 - val_accuracy: 0.7048 - val_loss: 8.0294
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.6506 - loss: 7.9875 - val_accuracy: 0.7744 - val_loss: 7.6365
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7209 - loss: 7.7044 - val_accuracy: 0.8143 - val_loss: 7.3721
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-25 10:35:43,587] Trial 0 finished with value: 0.8143333196640015 and parameters: {'lr': 5.097754401549031e-06, 'n_trials': 3, 'neurons_0': 704, 'activation_fn_0': 'leaky_relu', 'padding_0': 'valid', 'dropouts_0': 0.30000000000000004, 'kernel_initializer_0': 'he_normal', 'regularizer_0': 'l2', 'reg_rate_0': 0.00472187265996439, 'bias_initializer_0': 'glorot_uniform', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 0.0009579970448368154, 'neurons_1': 16, 'activation_fn_1': 'elu', 'padding_1': 'same', 'dropouts_1': 0.1, 'kernel_initializer_1': 'glorot_normal', 'regularizer_1': 'l2', 'reg_rate_1': 1.2880329475000244e-08, 'bias_initializer_1': 'he_uniform', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 7.29607939875754e-08, 'neurons_2': 320, 'activation_fn_2': 'relu', 'padding_2': 'same', 'dropouts_2': 0.2, 'kernel_initializer_2': 'glorot_normal', 'regularizer_2': 'l2', 'reg_rate_2': 0.00027556861813161205, 'bias_initializer_2': 'glorot_normal', 'bias_regularizer_2': 'l2', 'bias_

Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.8672 - loss: 0.9597 - val_accuracy: 0.9621 - val_loss: 0.3286
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9630 - loss: 0.2730 - val_accuracy: 0.9782 - val_loss: 0.1952
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9717 - loss: 0.2029 - val_accuracy: 0.9815 - val_loss: 0.1720
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9758 - loss: 0.1855 - val_accuracy: 0.9857 - val_loss: 0.1546
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9790 - loss: 0.1711 - val_accuracy: 0.9858 - val_loss: 0.1541
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-25 10:36:15,351] Trial 1 finished with value: 0.9858333468437195 and parameters: {'lr': 0.004209614222083942, 'n_trials': 2, 'neurons_0': 208, 'activation_fn_0': 'leaky_relu', 'padding_0': 'same', 'dropouts_0': 0.30000000000000004, 'kernel_initializer_0': 'glorot_normal', 'regularizer_0': 'l2', 'reg_rate_0': 0.0004930557609719655, 'bias_initializer_0': 'he_normal', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 0.000719023823122171, 'neurons_1': 80, 'activation_fn_1': 'tanh', 'padding_1': 'valid', 'dropouts_1': 0.4, 'kernel_initializer_1': 'he_normal', 'regularizer_1': 'l1', 'reg_rate_1': 0.00016240174473398632, 'bias_initializer_1': 'he_normal', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 1.1520149354842499e-05, 'temp_neurons': 192, 'temp_activation_fn': 'leaky_relu', 'epsilon': 4.5586483927015066e-07}. Best is trial 1 with value: 0.9858333468437195.


...

[I 2026-09-25 10:42:54,368] Trial 7 finished with value: 0.7989166378974915 and parameters: {'lr': 4.733350766170528e-05, 'n_trials': 1, 'neurons_0': 432, 'activation_fn_0': 'sigmoid', 'padding_0': 'same', 'dropouts_0': 0.30000000000000004, 'kernel_initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 3.0373707662164195e-07, 'bias_initializer_0': 'he_uniform', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 7.821053063374341e-06, 'temp_neurons': 160, 'temp_activation_fn': 'tanh', 'epsilon': 2.5719274260516823e-07}. Best is trial 2 with value: 0.9882500171661377.


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.2138 - loss: 2.2364 - val_accuracy: 0.4693 - val_loss: 2.1156
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.4347 - loss: 2.0514 - val_accuracy: 0.6653 - val_loss: 1.9320
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.5345 - loss: 1.8931 - val_accuracy: 0.6479 - val_loss: 1.7785
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.5931 - loss: 1.7592 - val_accuracy: 0.6569 - val_loss: 1.6518
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.6339 - loss: 1.6480 - val_accuracy: 0.6883 - val_loss: 1.5530
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-25 10:43:27,619] Trial 8 finished with value: 0.6882500052452087 and parameters: {'lr': 2.825132732263843e-06, 'n_trials': 1, 'neurons_0': 336, 'activation_fn_0': 'sigmoid', 'padding_0': 'same', 'dropouts_0': 0.4, 'kernel_initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 7.127003010750598e-08, 'bias_initializer_0': 'glorot_uniform', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 0.00020160948529767165, 'temp_neurons': 48, 'temp_activation_fn': 'sigmoid', 'epsilon': 7.474202123044666e-05}. Best is trial 2 with value: 0.9882500171661377.


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8658 - loss: 0.6324 - val_accuracy: 0.9348 - val_loss: 0.2925
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9392 - loss: 0.2514 - val_accuracy: 0.9546 - val_loss: 0.1878
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9562 - loss: 0.1720 - val_accuracy: 0.9657 - val_loss: 0.1388
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9680 - loss: 0.1279 - val_accuracy: 0.9715 - val_loss: 0.1139
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9747 - loss: 0.1001 - val_accuracy: 0.9771 - val_loss: 0.0922
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-25 10:43:44,155] Trial 9 finished with value: 0.9770833253860474 and parameters: {'lr': 0.0007908204283245202, 'n_trials': 1, 'neurons_0': 96, 'activation_fn_0': 'relu', 'padding_0': 'valid', 'dropouts_0': 0.1, 'kernel_initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 2.3277713794128827e-08, 'bias_initializer_0': 'glorot_normal', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 0.0007111192936669716, 'temp_neurons': 64, 'temp_activation_fn': 'sigmoid', 'epsilon': 0.0018789661678962548}. Best is trial 2 with value: 0.9882500171661377.


In [28]:
print(study.best_trial.number)
print()
study.best_params

2



{'lr': 0.00021796095395365246,
 'n_trials': 3,
 'neurons_0': 560,
 'activation_fn_0': 'leaky_relu',
 'padding_0': 'same',
 'dropouts_0': 0.1,
 'kernel_initializer_0': 'he_normal',
 'regularizer_0': 'l1',
 'reg_rate_0': 3.6713828109473024e-06,
 'bias_initializer_0': 'he_normal',
 'bias_regularizer_0': 'l1',
 'bias_reg_rate_0': 1.3641290375734348e-05,
 'neurons_1': 464,
 'activation_fn_1': 'leaky_relu',
 'padding_1': 'valid',
 'dropouts_1': 0.4,
 'kernel_initializer_1': 'glorot_normal',
 'regularizer_1': 'l1',
 'reg_rate_1': 1.3098072772840402e-05,
 'bias_initializer_1': 'glorot_uniform',
 'bias_regularizer_1': 'l1',
 'bias_reg_rate_1': 0.00032340785777271094,
 'neurons_2': 160,
 'activation_fn_2': 'relu',
 'padding_2': 'valid',
 'dropouts_2': 0.5,
 'kernel_initializer_2': 'he_normal',
 'regularizer_2': 'l2',
 'reg_rate_2': 1.4189410707422369e-05,
 'bias_initializer_2': 'glorot_uniform',
 'bias_regularizer_2': 'l1',
 'bias_reg_rate_2': 0.00016622166555405735,
 'temp_neurons': 16,
 'temp_

In [30]:
num = study.best_trial.number

model = keras.models.load_model(f'model_trial_{num}.keras')

In [32]:
model.fit(
    x_train, y_train,
    epochs=20,
    validation_split=0.2,
    verbose = 1,
    batch_size=128
)

Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9806 - loss: 0.3021 - val_accuracy: 0.9892 - val_loss: 0.2686
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9832 - loss: 0.2752 - val_accuracy: 0.9906 - val_loss: 0.2419
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9852 - loss: 0.2477 - val_accuracy: 0.9908 - val_loss: 0.2206
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9855 - loss: 0.2275 - val_accuracy: 0.9908 - val_loss: 0.2039
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9864 - loss: 0.2063 - val_accuracy: 0.9902 - val_loss: 0.1876
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9862 - loss: 0.1911 - val_accuracy: 0.9901 - val_loss: 0.1723
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9877 - loss: 0.1734 - val_accuracy: 0.9915 - val_loss: 0.1580
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 66ms/step - accuracy: 0.9888 - loss: 0.1583 - 

In [33]:
pred = model.predict(x_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step


In [35]:
# Finding the index (0-9) with the highest probability for each image
predicted_classes = np.argmax(pred, axis=1)

# Comparing predicted labels to actual y_test labels
accuracy = np.mean(predicted_classes == y_test)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 99.24%


In [36]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9924 - loss: 0.0806
Test Accuracy: 99.24%
Test Loss: 8.06%


In [37]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

predicted_classes = np.argmax(pred, axis=1)

accuracy = accuracy_score(y_test, predicted_classes)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")


print()
print("Classification Report:")
print(classification_report(y_test, predicted_classes))

print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, predicted_classes))

Test Accuracy: 99.24%


Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       980
           1       1.00      1.00      1.00      1135
           2       0.99      1.00      0.99      1032
           3       1.00      1.00      1.00      1010
           4       1.00      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       0.99      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000


Confusion Matrix:
[[ 977    0    0    0    0    0    1    1    1    0]
 [   0 1131    1    0    0    0    2    1    0    0]
 [   1    1 1028    0    0    0    0    1    1    0]
 [   0    0    0